# Module 08: Computational Research Design & Validation Lab
Interactive execution of the **8-Question Scientific Paper Audit Protocol**, **Statistical Power & Benjamini-Hochberg FDR Validation (Track A)**, and **Generative ABM Sensitivity vs. Null Baseline Comparison (Track B)**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# ------------------------------------------------------------------------------
# 1. STANDARDIZED 8-QUESTION SCIENTIFIC PAPER AUDIT DATABASE
# ------------------------------------------------------------------------------

PAPER_AUDITS = {
    '1. Schreiber (2000) - Transfer Entropy': [
        '1. Problem: Symmetric mutual information cannot distinguish directional information flow from shared static correlation.',
        '2. Hypothesis: Conditioning on target past history isolates directed predictive information T_{Y->X}.',
        '3. Model: Conditional mutual information I(Y_past; X_future | X_past) estimated via kernel or k-NN estimators.',
        '4. Assumptions: Local stationarity, Markov order k, and absence of unobserved common drivers.',
        '5. Key Result: Asymmetric coupling detection in coupled chaotic maps and physiological bivariate series.',
        '6. Reproducibility Target: Elimination of spurious links in Common Driver & Cascade motifs via Conditional TE.',
        '7. Limitations: Bivariate TE fails under confounders; high-dimensional conditioning suffers curse of dimensionality.',
        '8. Open Question: How does iterative greedy conditioning scale under non-stationary regime shifts?'
    ],
    '2. Hanson & Crutchfield (1997) - Rule 54 Particles': [
        '1. Problem: How to formally extract emergent computational structures from raw cellular automata spacetime patterns.',
        '2. Hypothesis: Regular spacetime domains act as a vacuum; localized defects behave as information-carrying particles.',
        '3. Model: Finite-State Transducer (FST) domain filters constructed from regular language grammar of Rule 54.',
        '4. Assumptions: Known periodic ether background and deterministic synchronous lattice updates.',
        '5. Key Result: Complete particle taxonomy (alpha, beta, gamma+, gamma-) and deterministic collision algebra.',
        '6. Reproducibility Target: Filtering period-4 ether to reveal particle tracks and collision vertices.',
        '7. Limitations: Domain filters are rule-specific and do not automatically generalize to arbitrary chaotic rules.',
        '8. Open Question: How robust is particle collision logic under stochastic update noise?'
    ],
    '3. Mitchell, Crutchfield & Hraber (1993/1994) - Evolved CA': [
        '1. Problem: Can evolutionary search discover decentralized local rules that solve a global density task?',
        '2. Hypothesis: GA discovers particle-based information propagation, though symmetry breaking impedes search.',
        '3. Model: Radius-3 1D CA (N=149) evolved via Genetic Algorithm on uniform-density initial configurations.',
        '4. Assumptions: Fitness evaluated over finite random initial states without explicit global communication.',
        '5. Key Result: Four evolutionary innovation epochs transitioning from naive block expansion to particle-based logic.',
        '6. Reproducibility Target: Accuracy gap between Epoch 1 local majority and GKL particle logic across uniform vs unbiased ICs.',
        '7. Limitations: No 2-state radius-3 CA achieves 100% accuracy on all configurations (Land & Belew impossibility).',
        '8. Open Question: How does co-evolving the initial configuration distribution alter evolutionary escape from local optima?'
    ],
    '4. Kauffman / Greil & Drossel (2005) - RBN Attractors': [
        '1. Problem: How do gene regulatory network topology (K) and update timing govern dynamical stability and cell fates?',
        '2. Hypothesis: Critical networks (s = 2Kp(1-p) = 1) balance homeostatic stability with adaptive responsiveness.',
        '3. Model: Random Boolean Networks with N nodes, in-degree K, Boolean bias p, and sync vs async updates.',
        '4. Assumptions: Annealed approximation for Derrida curve; uniform random wiring without modular hierarchy.',
        '5. Key Result: Superpolynomial attractor scaling in sync critical RBNs vs power-law scaling under random async updates.',
        '6. Reproducibility Target: Derrida damage spreading curve and sync vs async attractor convergence comparison.',
        '7. Limitations: Exhaustive state enumeration is restricted to small N; random sampling undercounts rare attractors.',
        '8. Open Question: How do biologically realistic canalizing functions reshape asynchronous attractor stability?'
    ],
    '5. Tero et al. (2010) - Physarum Tokyo Rail': [
        '1. Problem: Can a decentralized tubular organism design transport networks balancing cost, efficiency, and fault tolerance?',
        '2. Hypothesis: Positive feedback between flux Q_ij and tube conductance D_ij self-organizes near-optimal planar graphs.',
        '3. Model: Kirchhoff flow network coupled with adaptive ODE dD_ij/dt = |Q_ij|^gamma / (1 + |Q_ij|^gamma) - D_ij.',
        '4. Assumptions: Quasi-steady laminar flow and sequential random source-sink demand pairs.',
        '5. Key Result: Reconstructed networks match Tokyo rail efficiency and fault tolerance at comparable total length.',
        '6. Reproducibility Target: Multi-objective comparison (Cost, Global Efficiency, Fault Tolerance) against MST baseline.',
        '7. Limitations: Dynamic edge-cutting self-healing is an experimental model extension beyond static topology comparison.',
        '8. Open Question: How does non-uniform traffic demand weighting alter optimal feedback exponent gamma?'
    ],
    '6. Arthur (1994) & Challet-Zhang (1997) - El Farol & MG': [
        '1. Problem: How do boundedly rational agents coordinate when the optimal action depends on being in the minority?',
        '2. Hypothesis: Inductive competition among heterogeneous predictors self-organizes attendance near capacity.',
        '3. Model: Agents select highest-scoring local predictor from a private bag of S strategies using shared history.',
        '4. Assumptions: Agents act independently without direct social communication or collusion in the baseline model.',
        '5. Key Result: Attendance fluctuates around capacity (El Farol) and exhibits a phase transition in volatility at alpha_c.',
        '6. Reproducibility Target: Attendance stabilization near L=60 and volatility scaling across control parameter alpha.',
        '7. Limitations: Adding social network coupling breaks independence and triggers herding volatility.',
        '8. Open Question: What network topologies prevent herding cascades while preserving distributed information aggregation?'
    ],
}

# ------------------------------------------------------------------------------
# 2. VALIDATION SIMULATIONS: TRACK A (FDR POWER) & TRACK B (NULL MODEL BASELINE)
# ------------------------------------------------------------------------------

def benjamini_hochberg_fdr(p_values, alpha=0.05):
    '''Applies Benjamini-Hochberg False Discovery Rate control to an array of p-values.'''
    m = len(p_values)
    order = np.argsort(p_values)
    ranked_p = p_values[order]
    thresholds = alpha * (np.arange(1, m + 1) / float(m))
    passed = ranked_p <= thresholds
    reject = np.zeros(m, dtype=bool)
    if np.any(passed):
        max_k = int(np.max(np.where(passed)[0]))
        reject[order[: max_k + 1]] = True
    return reject

def simulate_fdr_validation(n_tests=50, true_edges=8, signal_snr=0.45, sample_size=300, seed=42):
    '''Simulates multiple-hypothesis network inference with and without BH-FDR control.'''
    rng = np.random.default_rng(seed)
    # Generate z-scores for true edges vs null non-edges
    eff = signal_snr * np.sqrt(sample_size / 100.0)
    z_true = rng.normal(loc=eff, scale=1.0, size=true_edges)
    z_null = rng.normal(loc=0.0, scale=1.0, size=n_tests - true_edges)
    z_all = np.concatenate([z_true, z_null])
    # One-sided normal survival function approximation via erfc
    from scipy.special import erfc
    p_vals = 0.5 * erfc(z_all / np.sqrt(2.0))

    raw_reject = p_vals < 0.05
    fdr_reject = benjamini_hochberg_fdr(p_vals, alpha=0.05)

    raw_tpr = np.mean(raw_reject[:true_edges])
    raw_fpr = np.mean(raw_reject[true_edges:])
    fdr_tpr = np.mean(fdr_reject[:true_edges])
    fdr_fpr = np.mean(fdr_reject[true_edges:])
    return (raw_tpr, raw_fpr), (fdr_tpr, fdr_fpr)

def simulate_abm_vs_null(coupling_strength=0.6, n_seeds=15, seed=42):
    '''Compares structured local interaction model against randomized null baseline.'''
    rng = np.random.default_rng(seed)
    params = np.linspace(0.0, 1.0, 11)
    model_means, model_stds = [], []
    null_means, null_stds = [], []

    for p in params:
        # Nonlinear cooperative emergence curve with finite-sample seed variance
        m_vals = 1.0 / (1.0 + np.exp(-10.0 * (p - 0.45))) + rng.normal(0.0, 0.05, size=n_seeds)
        # Randomized Null Model baseline (destroys local structure)
        n_vals = 0.15 + 0.05 * p + rng.normal(0.0, 0.04, size=n_seeds)
        model_means.append(np.mean(np.clip(m_vals, 0, 1)))
        model_stds.append(np.std(m_vals))
        null_means.append(np.mean(np.clip(n_vals, 0, 1)))
        null_stds.append(np.std(n_vals))

    return params, np.array(model_means), np.array(model_stds), np.array(null_means), np.array(null_stds)

# ------------------------------------------------------------------------------
# 3. INTERACTIVE MODULE 08 RESEARCH DESIGN & VALIDATION LAB
# ------------------------------------------------------------------------------

def run_module08_lab(paper_choice='1. Schreiber (2000) - Transfer Entropy', track='Track A: Observational FDR & Power', sample_size=300):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    fig.patch.set_facecolor('#F8FAFC')

    if track == 'Track A: Observational FDR & Power':
        (r_tpr, r_fpr), (f_tpr, f_fpr) = simulate_fdr_validation(50, 8, 0.55, sample_size, seed=42)
        axes[0].set_facecolor('#F8FAFC')
        cats = ['True Positive Rate\n(Statistical Power)', 'False Positive Rate\n(Spurious Edges)']
        x = np.arange(2)
        axes[0].bar(x - 0.17, [r_tpr, r_fpr], width=0.34, color='#E11D48', edgecolor='#0F172A', label='Uncorrected (p < 0.05)')
        axes[0].bar(x + 0.17, [f_tpr, f_fpr], width=0.34, color='#0D9488', edgecolor='#0F172A', label='BH-FDR Corrected (q < 0.05)')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(cats, fontweight='bold')
        axes[0].set_ylim(0, 1.1)
        axes[0].set_title(f'1. Multiple Testing Control (N_samples = {sample_size})', fontsize=11, fontweight='bold')
        axes[0].legend(facecolor='white')
        axes[0].grid(axis='y', linestyle='--', alpha=0.4)

        # Power curve across sample sizes
        t_grid = [100, 200, 300, 500, 800, 1200]
        fdr_powers = [simulate_fdr_validation(50, 8, 0.55, t, seed=42)[1][0] for t in t_grid]
        axes[1].set_facecolor('#F8FAFC')
        axes[1].plot(t_grid, fdr_powers, marker='o', color='#2563EB', linewidth=2.2, label='BH-FDR Power')
        axes[1].axhline(0.80, color='#D97706', linestyle='--', label='80% Power Benchmark')
        axes[1].set_ylim(0, 1.05)
        axes[1].set_xlabel('Time-Series Sample Length T')
        axes[1].set_ylabel('Detection Power')
        axes[1].set_title('2. Minimum Sample Size Calibration Curve', fontsize=11, fontweight='bold')
        axes[1].legend(facecolor='white')
        axes[1].grid(True, linestyle='--', alpha=0.4)
    else:
        params, m_mean, m_std, n_mean, n_std = simulate_abm_vs_null(0.6, n_seeds=15, seed=42)
        axes[0].set_facecolor('#F8FAFC')
        axes[0].plot(params, m_mean, marker='o', color='#2563EB', linewidth=2.2, label='Mechanistic Model (Mean +/- 1 SD)')
        axes[0].fill_between(params, m_mean - m_std, m_mean + m_std, color='#2563EB', alpha=0.2)
        axes[0].plot(params, n_mean, marker='s', color='#64748B', linestyle='--', linewidth=2.0, label='Randomized Null Baseline')
        axes[0].fill_between(params, n_mean - n_std, n_mean + n_std, color='#64748B', alpha=0.2)
        axes[0].set_ylim(0, 1.1)
        axes[0].set_xlabel('Control Coupling Parameter')
        axes[0].set_ylabel('Emergent Macroscopic Metric')
        axes[0].set_title('1. Sensitivity Sweep vs. Null Model Baseline', fontsize=11, fontweight='bold')
        axes[0].legend(facecolor='white')
        axes[0].grid(True, linestyle='--', alpha=0.4)

        # Effect size (Cohen's d) over null baseline
        cohen_d = (m_mean - n_mean) / np.sqrt(0.5 * (m_std ** 2 + n_std ** 2) + 1e-9)
        axes[1].set_facecolor('#F8FAFC')
        axes[1].bar(params, cohen_d, width=0.07, color='#0D9488', edgecolor='#0F172A')
        axes[1].set_xlabel('Control Coupling Parameter')
        axes[1].set_ylabel('Standardized Effect Size (Cohen d)')
        axes[1].set_title('2. Falsifiability & Effect Separation Profile', fontsize=11, fontweight='bold')
        axes[1].grid(axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.show()

    print('=' * 88)
    print(f'8-QUESTION PAPER AUDIT PROTOCOL | {paper_choice}')
    print('=' * 88)
    for line in PAPER_AUDITS[paper_choice]:
        print(line)
    print('=' * 88)

widgets.interact(
    run_module08_lab,
    paper_choice=widgets.Dropdown(options=list(PAPER_AUDITS.keys()), value='1. Schreiber (2000) - Transfer Entropy', description='Paper Audit:'),
    track=widgets.Dropdown(options=['Track A: Observational FDR & Power', 'Track B: Generative ABM vs Null Model'], value='Track A: Observational FDR & Power', description='Validation:'),
    sample_size=widgets.IntSlider(min=100, max=1200, step=100, value=300, description='Sample T:'),
);
